# Lighting Persona Transformer Training (Google Colab)\n
\n
Use this notebook in **Google Colab Pro/Pro+** to train the lighting persona Transformer on GPU.\n
\n
Expected input file: `lightningData.csv`\n
\n
Outputs match the local project filenames:\n
\n
- `lighting_persona_transformer.pt`\n
- `lighting_persona_transformer_metadata.json`\n
- `lighting_persona_transformer_report.txt`\n
- `lighting_persona_transformer_confusion_matrix.csv`\n
- `lighting_persona_transformer_outputs.zip`\n

## 1. Runtime\n
\n
Before running: `Runtime` → `Change runtime type` → choose **GPU**.

In [9]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-80GB


## 2. Provide `lightningData.csv`\n
\n
Choose one option.\n
\n
**Option A:** upload `lightningData.csv` directly to Colab.\n
\n
**Option B:** mount Google Drive and set `DATA_CSV` to the Drive path.

In [11]:
# Google Drive mode. Run this cell if your dataset is in Drive.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# If you know the exact path, set it here. Otherwise leave as None and the notebook will search MyDrive.
KNOWN_DATA_CSV = None
# Example:
# KNOWN_DATA_CSV = '/content/drive/MyDrive/VisualizationApp/Data/lightningData.csv'

if KNOWN_DATA_CSV:
    DATA_CSV = KNOWN_DATA_CSV
else:
    matches = list(Path('/content/drive/MyDrive/VisualizationApp').rglob('lightningData.csv'))
    if not matches:
        raise FileNotFoundError('Could not find lightningData.csv in /content/drive/MyDrive/VisualizationApp. Set KNOWN_DATA_CSV to the exact Drive path.')
    print('Found CSV files:')
    for i, match in enumerate(matches):
        print(i, match)
    DATA_CSV = str(matches[0])

OUT_DIR = '/content/lighting_persona_transformer_outputs'

print('DATA_CSV =', DATA_CSV)
print('OUT_DIR =', OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found CSV files:
0 /content/drive/MyDrive/VisualizationApp/lightningData.csv
DATA_CSV = /content/drive/MyDrive/VisualizationApp/lightningData.csv
OUT_DIR = /content/lighting_persona_transformer_outputs


## 3. Training Configuration\n
\n
For Colab Pro+, the defaults below should be reasonable. If GPU memory is limited, reduce `BATCH_SIZE`, `D_MODEL`, or `N_LAYERS`.

In [12]:
MAX_ROWS = None       # Use None for full dataset. Use e.g. 500000 for a faster test.\n
EPOCHS = 35
BATCH_SIZE = 64
D_MODEL = 96
N_HEADS = 4
N_LAYERS = 3
DIM_FEEDFORWARD = 192
DROPOUT = 0.15
LR = 3e-4
WEIGHT_DECAY = 1e-2
TRAIN_FRACTION = 0.8
SEED = 42

In [13]:
import json
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

SEQ_LEN = 288
DEFAULT_LAMPS = [
    'bed_left', 'bed_right', 'cabinet', 'closet', 'corridor_left',
    'corridor_right', 'dinner_table', 'hidden_top', 'shower', 'sink', 'table',
]

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def dominant_value(series):
    modes = series.dropna().mode()
    return modes.iat[0] if len(modes) else np.nan

def build_room_day_sequences(data_csv, max_rows=None):
    usecols = [
        'timestamp', 'room_number', 'lamp_location', 'Value',
        'reservation_active', 'pir_motion', 'n_occupants', 'active_actors',
        'hurry_morning', 'lazy_day', 'forgetful', 'lightning_persona',
    ]
    df = pd.read_csv(data_csv, usecols=usecols, parse_dates=['timestamp'], nrows=max_rows)
    df = df[df['reservation_active'].eq('Yes')].copy()
    df = df.dropna(subset=['timestamp', 'lightning_persona'])
    df['date'] = df['timestamp'].dt.date
    df['sample_id'] = df['room_number'].astype(str) + '|' + df['date'].astype(str)
    df['slot'] = df['timestamp'].dt.hour * 12 + df['timestamp'].dt.minute // 5
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce').fillna(0).clip(0, 80) / 80.0

    for col in ['pir_motion', 'n_occupants', 'active_actors', 'hurry_morning', 'lazy_day', 'forgetful']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    target = df.groupby('sample_id')['lightning_persona'].agg(dominant_value).dropna()
    sample_ids = target.index.tolist()
    slot_index = pd.MultiIndex.from_product([sample_ids, range(SEQ_LEN)], names=['sample_id', 'slot'])

    lamp_df = df[df['lamp_location'].ne('none')].copy()
    lamp_profile = (
        lamp_df.groupby(['sample_id', 'slot', 'lamp_location'])['Value']
        .mean()
        .unstack(fill_value=0.0)
        .reindex(columns=DEFAULT_LAMPS, fill_value=0.0)
        .reindex(slot_index, fill_value=0.0)
    )

    context_cols = ['pir_motion', 'n_occupants', 'active_actors', 'hurry_morning', 'lazy_day', 'forgetful']
    context_profile = (
        df.groupby(['sample_id', 'slot'])[context_cols]
        .mean()
        .reindex(slot_index, fill_value=0.0)
    )

    slots = np.tile(np.arange(SEQ_LEN), len(sample_ids))
    time_profile = pd.DataFrame(
        {
            'hour_sin': np.sin(2 * np.pi * (slots // 12) / 24),
            'hour_cos': np.cos(2 * np.pi * (slots // 12) / 24),
        },
        index=slot_index,
    )

    features = pd.concat([lamp_profile, context_profile, time_profile], axis=1)
    feature_names = list(features.columns)
    x = features.to_numpy(dtype=np.float32).reshape(len(sample_ids), SEQ_LEN, len(feature_names))
    y = target.to_numpy()
    return x, y, feature_names

In [14]:
class LightingPersonaTransformer(nn.Module):
    def __init__(self, input_dim, n_classes, seq_len=SEQ_LEN, d_model=96, n_heads=4, n_layers=3, dim_feedforward=192, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        hidden = self.input_proj(x) + self.pos_embed
        hidden = self.encoder(hidden)
        hidden = self.norm(hidden)
        pooled = hidden.mean(dim=1)
        return self.head(pooled)

def chronological_split(x, y, train_fraction=0.8):
    split_idx = max(1, min(len(x) - 1, int(len(x) * train_fraction)))
    return x[:split_idx], x[split_idx:], y[:split_idx], y[split_idx:]

## 4. Train

In [15]:
set_seed(SEED)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

x, y_str, feature_names = build_room_day_sequences(DATA_CSV, max_rows=MAX_ROWS)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_str)
classes = list(label_encoder.classes_)

x_train, x_test, y_train, y_test = chronological_split(x, y, train_fraction=TRAIN_FRACTION)
counts = np.bincount(y_train, minlength=len(classes)).clip(min=1)
weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('samples:', len(x), 'train:', len(x_train), 'test:', len(x_test))
print('classes:', classes)
print('input shape:', x.shape)

model = LightingPersonaTransformer(
    input_dim=x.shape[-1],
    n_classes=len(classes),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.CrossEntropyLoss(weight=weights.to(device))

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train).long()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
x_test_tensor = torch.from_numpy(x_test).to(device)

best_acc = -1.0
best_state = None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)

    model.eval()
    with torch.no_grad():
        pred = model(x_test_tensor).argmax(dim=1).cpu().numpy()
    acc = float((pred == y_test).mean())
    if acc > best_acc:
        best_acc = acc
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f'epoch {epoch:03d} train_loss={total_loss / len(x_train):.4f} test_acc={acc:.3f}')

print('best test accuracy:', best_acc)

device: cuda
samples: 9132 train: 7305 test: 1827
classes: ['Balanced', 'NightFocused', 'Routine', 'StaticBright', 'StaticDim']
input shape: (9132, 288, 19)


/tmp/ipykernel_50218/1950736031.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


epoch 001 train_loss=1.2659 test_acc=0.529
epoch 005 train_loss=0.4913 test_acc=0.817
epoch 010 train_loss=0.3441 test_acc=0.870
epoch 015 train_loss=0.2570 test_acc=0.899
epoch 020 train_loss=0.1956 test_acc=0.915
epoch 025 train_loss=0.1688 test_acc=0.918
epoch 030 train_loss=0.1281 test_acc=0.914
epoch 035 train_loss=0.0972 test_acc=0.918
best test accuracy: 0.9250136836343733


## 5. Save Outputs

In [16]:
model.load_state_dict(best_state)
model.to(device)
model.eval()
with torch.no_grad():
    y_pred = model(x_test_tensor).argmax(dim=1).cpu().numpy()

report = classification_report(
    y_test,
    y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    digits=3,
    zero_division=0,
)
confusion = confusion_matrix(y_test, y_pred, labels=range(len(classes)))
cm_df = pd.DataFrame(confusion, index=classes, columns=classes)
cm_df.index.name = 'true'

out_dir = Path(OUT_DIR)
model_path = out_dir / 'lighting_persona_transformer.pt'
metadata_path = out_dir / 'lighting_persona_transformer_metadata.json'
report_path = out_dir / 'lighting_persona_transformer_report.txt'
cm_path = out_dir / 'lighting_persona_transformer_confusion_matrix.csv'

report_text = '\n'.join([
    f'samples: {len(x):,}',
    f'train samples: {len(x_train):,}',
    f'test samples: {len(x_test):,}',
    f'sequence length: {SEQ_LEN}',
    f'input features: {len(feature_names)}',
    f'best test accuracy: {best_acc:.3f}',
    '',
    report,
])
report_path.write_text(report_text)
cm_df.to_csv(cm_path)
torch.save(
    {
        'model_state_dict': best_state,
        'classes': classes,
        'feature_names': feature_names,
        'seq_len': SEQ_LEN,
        'input_dim': x.shape[-1],
        'config': {
            'd_model': D_MODEL,
            'n_heads': N_HEADS,
            'n_layers': N_LAYERS,
            'dim_feedforward': DIM_FEEDFORWARD,
            'dropout': DROPOUT,
        },
    },
    model_path,
)
metadata_path.write_text(json.dumps({
    'classes': classes,
    'feature_names': feature_names,
    'seq_len': SEQ_LEN,
    'input_dim': x.shape[-1],
    'model_file': model_path.name,
    'report_file': report_path.name,
    'confusion_matrix_file': cm_path.name,
}, indent=2))

print(report_text)
print('saved:', model_path)
print('saved:', metadata_path)
print('saved:', report_path)
print('saved:', cm_path)

samples: 9,132
train samples: 7,305
test samples: 1,827
sequence length: 288
input features: 19
best test accuracy: 0.925

              precision    recall  f1-score   support

    Balanced      0.917     0.880     0.898       451
NightFocused      0.901     0.939     0.920       213
     Routine      0.910     0.956     0.933       551
StaticBright      0.960     0.897     0.928       321
   StaticDim      0.949     0.955     0.952       291

    accuracy                          0.925      1827
   macro avg      0.927     0.926     0.926      1827
weighted avg      0.926     0.925     0.925      1827

saved: /content/lighting_persona_transformer_outputs/lighting_persona_transformer.pt
saved: /content/lighting_persona_transformer_outputs/lighting_persona_transformer_metadata.json
saved: /content/lighting_persona_transformer_outputs/lighting_persona_transformer_report.txt
saved: /content/lighting_persona_transformer_outputs/lighting_persona_transformer_confusion_matrix.csv


In [18]:
drive_zip_path = Path('/content/drive/MyDrive/lighting_persona_transformer_outputs.zip')

with zipfile.ZipFile(drive_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [model_path, metadata_path, report_path, cm_path]:
        zf.write(path, arcname=path.name)

print('saved to Drive:', drive_zip_path)

saved to Drive: /content/drive/MyDrive/lighting_persona_transformer_outputs.zip


## 6. Copy Outputs Back Locally\n
\n
After downloading and unzipping `lighting_persona_transformer_outputs.zip`, copy the files into:\n
\n
`AIModelsAndAlgorithms/LightingPersona/transformer/`\n
\n
Then tell Codex you trained the Colab Transformer model and want it wired into the backend.